# **Probabilidades con modelo Naive Bayes**
### **AUTOR:** HADSON PAREDES
### **SOLUCIÓN:** Agente de Aprendizaje Generativo (Naive Bayes) para Clasificación del Juego según Condiciones Climáticas

[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Networking: Linkedin](https://img.shields.io/badge/LinkedIn-Hadson%20Paredes-blue?logo=linkedin&style=flat)](https://www.linkedin.com/in/hadson-paredes/) 
[![Networking: Facebook](https://img.shields.io/badge/Facebook-Hadson%20Paredes%20Cordova-Gree?logo=facebook&style=flat)](https://www.facebook.com/hadson.paredescordova/) 
[![Networking: X](https://img.shields.io/badge/Hadson%20Paredes-black?logo=x&style=flat)](https://x.com/hadson_paredes)

## Resumen del caso

Se desea construir un **agente de aprendizaje generativo** que, a partir de condiciones climáticas observadas (`outlook`, `temperature`, `humidity`, `windy`), sea capaz de **predecir si se juega o no** (`play`). Un modelo generativo, a diferencia de uno discriminativo, no aprende directamente la frontera de decisión entre clases: en su lugar, **modela cómo se generan los datos dentro de cada clase**, es decir, aprende la distribución conjunta `P(X, Y)` a partir de las distribuciones marginales y condicionales, y luego aplica la regla de Bayes para inferir la clase más probable dado un conjunto de evidencias.

El enfoque elegido es un **clasificador Naive Bayes categórico**, adecuado porque:

- Todas las variables del dataset `weather.nominal.csv` son **categóricas nominales**.
- El tamaño de la muestra es pequeño (14 instancias), y Naive Bayes es robusto en estos escenarios al requerir estimar solo distribuciones univariadas condicionadas a la clase.
- Su naturaleza generativa permite explicar el problema en términos de probabilidad conjunta, marginal y condicional, cumpliendo el objetivo pedagógico de esta solución.

### Descripción Detallada de Variables

**`outlook`**: Representa la condición climática general del día. Es una variable categórica con los siguientes valores posibles:
*   `sunny` (soleado)
*   `overcast` (nublado)
*   `rainy` (lluvioso)

Esta variable es crucial para determinar si es un buen día para actividades al aire libre.

**`temperature`**: Indica la temperatura del día. Es una variable categórica con los siguientes niveles:
*   `hot` (caluroso)
*   `mild` (templado)
*   `cool` (frío)

La temperatura es un factor importante que influye en la comodidad para realizar actividades.

**`windy`**: Indica si el día es ventoso. Es una variable booleana:
*   `True` (sí, está ventoso)
*   `False` (no, no está ventoso)

El viento puede ser un impedimento para ciertas actividades al aire libre.

**`play`**: Es la variable objetivo, indicando si se decide realizar la actividad deportiva/al aire libre. Es una variable categórica binaria:
*   `yes` (sí, se juega/realiza la actividad)
*   `no` (no, no se juega/realiza la actividad)

El objetivo del modelo es predecir este valor basándose en las otras variables.

### La solución se desarrolla en tres grandes bloques:

1. **Análisis de correlación de variables** (prueba Chi-cuadrado de independencia) para seleccionar los predictores más relevantes respecto a `play`.
2. **Implementación del modelo probabilístico estilo Naive Bayes**, estimando explícitamente las distribuciones marginal, conjunta y condicional a partir de los datos de entrenamiento.
3. **Inferencia y validación**, aplicando la productoria de probabilidades condicionales (regla de decisión MAP de Naive Bayes) sobre los datos de prueba y evaluando el desempeño del modelo.


In [43]:
# Librerías necesarias
import numpy as np
import pandas as pd
from itertools import product

from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option('display.max_columns', None)
np.random.seed(42)


In [44]:
# Carga del dataset
df = pd.read_csv('weather.nominal.csv')

TARGET = 'play'
FEATURES = [c for c in df.columns if c != TARGET]

print(f"Dimensiones del dataset: {df.shape}")
print(f"Variable objetivo: '{TARGET}'  |  Variables predictoras: {FEATURES}")
df


Dimensiones del dataset: (14, 5)
Variable objetivo: 'play'  |  Variables predictoras: ['outlook', 'temperature', 'humidity', 'windy']


,outlook,temperature,humidity,windy,play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes
5,rainy,cool,normal,True,no
6,overcast,cool,normal,True,yes
7,sunny,mild,high,False,no
8,sunny,cool,normal,False,yes
9,rainy,mild,normal,False,yes


In [45]:
df.info()
print()
print("Distribución de la variable objetivo:")
print(df[TARGET].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   outlook      14 non-null     object
 1   temperature  14 non-null     object
 2   humidity     14 non-null     object
 3   windy        14 non-null     bool  
 4   play         14 non-null     object
dtypes: bool(1), object(4)
memory usage: 594.0+ bytes

Distribución de la variable objetivo:
play
yes    9
no     5
Name: count, dtype: int64


---
## 1. Análisis de correlación de variables

### 1.1 Cálculo de la relación entre cada variable y la variable objetivo (Chi-cuadrado)

Dado que todas las variables son **categóricas/nominales**, no es posible usar coeficientes de correlación lineales (como Pearson). En su lugar, se emplea la **prueba Chi-cuadrado de independencia (χ²)**, que evalúa si existe una asociación estadísticamente significativa entre cada variable predictora y la variable objetivo `play`.

Para cada variable predictora `X`:

1. Se construye la **tabla de contingencia** entre `X` y `play`.
2. Se calcula el estadístico χ² y su p-valor comparando las frecuencias observadas con las frecuencias esperadas bajo independencia.
3. Un **χ² alto** (y un p-valor bajo) indica que la variable **está asociada** con la variable objetivo (mayor relevancia predictiva); un χ² bajo indica poca o nula asociación.


In [46]:
def chi2_vs_target(df, feature, target):
    # Calcula la prueba Chi-cuadrado de independencia entre `feature` y `target`.
    tabla_contingencia = pd.crosstab(df[feature], df[target])
    chi2, p, dof, expected = chi2_contingency(tabla_contingencia, correction=False)
    return chi2, p, dof, tabla_contingencia

resultados = []
tablas = {}

for feat in FEATURES:
    chi2, p, dof, tabla = chi2_vs_target(df, feat, TARGET)
    tablas[feat] = tabla
    resultados.append({
        'variable': feat,
        'chi2': round(chi2, 4),
        'p_valor': round(p, 4),
        'grados_libertad': dof
    })

df_chi2 = pd.DataFrame(resultados).sort_values('chi2', ascending=False).reset_index(drop=True)
df_chi2


,variable,chi2,p_valor,grados_libertad
0,outlook,3.5467,0.1698,2
1,humidity,2.8000,0.0943,1
2,windy,0.9333,0.3340,1
3,temperature,0.5704,0.7519,2


In [47]:
# Tablas de contingencia (frecuencias observadas) usadas en el cálculo de Chi-cuadrado
for feat in FEATURES:
    print(f"--- Tabla de contingencia: {feat} vs {TARGET} ---")
    print(tablas[feat])
    print()


--- Tabla de contingencia: outlook vs play ---
play      no  yes
outlook          
overcast   0    4
rainy      2    3
sunny      3    2

--- Tabla de contingencia: temperature vs play ---
play         no  yes
temperature         
cool          1    3
hot           2    2
mild          2    4

--- Tabla de contingencia: humidity vs play ---
play      no  yes
humidity         
high       4    3
normal     1    6

--- Tabla de contingencia: windy vs play ---
play   no  yes
windy         
False   2    6
True    3    3



### 1.2 Selección de las variables más relevantes

El criterio de selección utilizado es la **magnitud del estadístico χ²**: a mayor valor de χ², mayor evidencia de asociación (dependencia) entre la variable predictora y `play`. Como el dataset es muy pequeño (14 instancias), los p-valores no siempre alcanzan significancia estadística estricta (p < 0.05); por ello se prioriza el **ordenamiento relativo del estadístico χ²** como medida de relevancia, en lugar de aplicar un corte binario rígido por p-valor.

Se seleccionan las variables cuyo χ² se ubique **por encima de la mediana** de los χ² calculados, garantizando conservar al menos dos predictores (requisito mínimo para que el supuesto de independencia condicional de Naive Bayes tenga sentido práctico).


In [48]:
umbral = df_chi2['chi2'].median()
seleccionadas = df_chi2[df_chi2['chi2'] >= umbral]['variable'].tolist()

# Garantizar un mínimo de 2 variables seleccionadas
if len(seleccionadas) < 2:
    seleccionadas = df_chi2['variable'].tolist()[:2]

print(f"Umbral (mediana de chi2): {umbral:.4f}")
print(f"Variables seleccionadas: {seleccionadas}")
df_chi2


Umbral (mediana de chi2): 1.8666
Variables seleccionadas: ['outlook', 'humidity']


,variable,chi2,p_valor,grados_libertad
0,outlook,3.5467,0.1698,2
1,humidity,2.8000,0.0943,1
2,windy,0.9333,0.3340,1
3,temperature,0.5704,0.7519,2


### 1.3 Construcción del dataset simplificado

Se construye un nuevo dataset que conserva únicamente las variables seleccionadas y la variable objetivo.


In [49]:
df_simplificado = df[seleccionadas + [TARGET]].copy()
print(f"Dataset simplificado -> columnas: {list(df_simplificado.columns)}")
df_simplificado

Dataset simplificado -> columnas: ['outlook', 'humidity', 'play']


,outlook,humidity,play
0,sunny,high,no
1,sunny,high,no
2,overcast,high,yes
3,rainy,high,yes
4,rainy,normal,yes
5,rainy,normal,no
6,overcast,normal,yes
7,sunny,high,no
8,sunny,normal,yes
9,rainy,normal,yes


### 1.4 Justificación de la selección

La tabla de resultados de la sección 1.1 muestra que las variables con mayor estadístico χ² son aquellas cuya distribución de valores difiere más marcadamente entre las clases `yes` y `no` de `play`. Concretamente:

- Las variables con **χ² alto** presentan tablas de contingencia donde ciertas categorías se concentran claramente en una sola clase (por ejemplo, `outlook = overcast` se asocia casi exclusivamente con `play = yes`), lo cual aporta **información discriminativa fuerte**.
- Las variables con **χ² bajo** (cercano a 0) presentan una distribución de categorías casi proporcional entre ambas clases de `play`, por lo que su capacidad para separar las clases es limitada.

Por ello, se retienen las variables ubicadas en la mitad superior del ranking de χ² (paso 1.2), ya que son las que **más información aportan** para discriminar entre `play = yes` y `play = no`, mientras se descartan las de menor asociación para reducir ruido y dimensionalidad del modelo generativo.


### 1.5 Estructura teórica de probabilidades (modelo Naive Bayes)

Sea `Y` la variable objetivo (`play`) y `X = (X₁, X₂, …, Xₙ)` el vector de variables predictoras seleccionadas. El modelo Naive Bayes es un **modelo generativo** que se apoya en tres tipos de distribuciones:

**a) Probabilidad marginal** — probabilidad de cada clase, sin considerar evidencia alguna:

$$P(Y = y) = \frac{\text{número de instancias con } Y = y}{N}$$

**b) Probabilidad conjunta** — probabilidad de observar simultáneamente la clase y una combinación de valores de las variables:

$$P(X_1, X_2, \dots, X_n, Y) = P(Y) \cdot P(X_1, X_2, \dots, X_n \mid Y)$$

**c) Probabilidad condicional** — probabilidad de observar un valor de una variable predictora dado que se conoce la clase:

$$P(X_i = x_i \mid Y = y) = \frac{\text{número de instancias con } Y=y \text{ y } X_i = x_i}{\text{número de instancias con } Y = y}$$

**Supuesto "naive" (independencia condicional):** Naive Bayes asume que, **dada la clase Y**, las variables predictoras son condicionalmente independientes entre sí. Esto permite factorizar la probabilidad conjunta condicional como un **producto de probabilidades condicionales individuales**, en lugar de tener que estimar la distribución conjunta completa (que requeriría muchísimos más datos):

$$P(X_1, X_2, \dots, X_n \mid Y) \approx \prod_{i=1}^{n} P(X_i \mid Y)$$

Aplicando el **teorema de Bayes**, la probabilidad posterior de la clase dado un vector de evidencia es:

$$P(Y \mid X_1, \dots, X_n) = \frac{P(Y)\prod_{i=1}^{n}P(X_i \mid Y)}{P(X_1,\dots,X_n)}$$

Como el denominador `P(X₁,…,Xₙ)` es constante para todas las clases (no depende de `Y`), la regla de decisión **MAP (Maximum A Posteriori)** que usará el modelo es:

$$\hat{y} = \arg\max_{y} \; P(Y=y)\prod_{i=1}^{n} P(X_i = x_i \mid Y=y)$$


### 1.6 Resumen descriptivo del análisis de correlación de variables

El análisis de correlación mediante la prueba **Chi-cuadrado de independencia** arrojó el siguiente ranking de asociación con `play`: **`outlook`** (χ² = 3.5467, p = 0.1698) en primer lugar, seguido de **`humidity`** (χ² = 2.8000, p = 0.0943), luego `windy` (χ² = 0.9333, p = 0.3340) y por último `temperature` (χ² = 0.5704, p = 0.7519), con el estadístico más bajo del conjunto.

Al observar las tablas de contingencia, `outlook` resultó ser el predictor más informativo: cuando `outlook = overcast`, el 100% de las instancias (4 de 4) corresponden a `play = yes`, mientras que `sunny` se inclina mayoritariamente hacia `play = no` (3 de 5). De forma similar, `humidity = normal` se asocia con `play = yes` en 6 de 7 casos, frente a una distribución mucho más pareja cuando `humidity = high` (3 "no" vs. 4 "yes"). En contraste, `windy` y, sobre todo, `temperature` muestran una distribución de categorías casi equilibrada entre ambas clases, lo que se traduce en un aporte discriminativo marginal.

Con base en este ordenamiento y aplicando el criterio de la mediana de χ² (umbral = 1.8666), se seleccionaron las variables **`outlook`** y **`humidity`** como predictores más relevantes, descartando `windy` y `temperature`. El dataset simplificado resultante conserva 14 instancias con solo estas dos variables predictoras más la variable objetivo `play`, reduciendo la dimensionalidad del problema sin perder las señales estadísticamente más fuertes. Adicionalmente, se estableció el marco teórico de probabilidad **marginal, conjunta y condicional** que sustentará la construcción del modelo Naive Bayes en la siguiente sección.


---
## 2. Implementación del modelo probabilístico (estilo Naive Bayes)

### 2.1 División en entrenamiento y prueba

Se divide el dataset simplificado en un subconjunto de **entrenamiento (70%)** y uno de **prueba (30%)**. Dado el tamaño reducido del dataset (14 filas), se fija una semilla aleatoria para garantizar reproducibilidad.


In [50]:
X = df_simplificado[seleccionadas]
y = df_simplificado[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=None
)

train_df = X_train.copy()
train_df[TARGET] = y_train

test_df = X_test.copy()
test_df[TARGET] = y_test

print(f"Tamaño entrenamiento: {train_df.shape[0]} instancias")
print(f"Tamaño prueba: {test_df.shape[0]} instancias")
print("\nConjunto de ENTRENAMIENTO:")
display(train_df)
print("\nConjunto de PRUEBA:")
display(test_df)

Tamaño entrenamiento: 9 instancias
Tamaño prueba: 5 instancias

Conjunto de ENTRENAMIENTO:


,outlook,humidity,play
8,sunny,normal,yes
2,overcast,high,yes
1,sunny,high,no
13,rainy,high,no
4,rainy,normal,yes
7,sunny,high,no
10,sunny,normal,yes
3,rainy,high,yes
6,overcast,normal,yes



Conjunto de PRUEBA:


,outlook,humidity,play
9,rainy,normal,yes
11,overcast,high,yes
0,sunny,high,no
12,overcast,normal,yes
5,rainy,normal,no


### 2.2 Estimación de las distribuciones de probabilidad (marginal, conjunta y condicional)

Con los datos de **entrenamiento**, se estiman:

- La **probabilidad marginal** de cada clase `P(play)`.
- La **probabilidad condicional** `P(Xᵢ = xᵢ | play)` para cada variable seleccionada y cada uno de sus valores posibles, usando **suavizado de Laplace** (add-one smoothing) para evitar probabilidades nulas ante categorías no observadas en entrenamiento para alguna clase.
- La **probabilidad conjunta** aproximada `P(Xᵢ, play) = P(play) · P(Xᵢ | play)`, como tabla de referencia.


In [51]:
clases = sorted(train_df[TARGET].unique())
N_train = len(train_df)

# --- Probabilidad MARGINAL de la clase: P(Y = y) ---
prob_marginal = train_df[TARGET].value_counts(normalize=True).reindex(clases)
print("Probabilidad marginal P(play):")
print(prob_marginal)

Probabilidad marginal P(play):
play
no     0.333333
yes    0.666667
Name: proportion, dtype: float64


In [52]:
# --- Probabilidad CONDICIONAL: P(Xi = xi | Y = y), con suavizado de Laplace ---
def calcular_condicionales(train_df, features, target, clases, alpha=1):
    condicionales = {}
    for feat in features:
        valores_posibles = sorted(train_df[feat].unique())
        k = len(valores_posibles)
        tabla = pd.DataFrame(index=valores_posibles, columns=clases, dtype=float)
        for y in clases:
            subset = train_df[train_df[target] == y]
            n_y = len(subset)
            conteo_valores = subset[feat].value_counts()
            for val in valores_posibles:
                conteo = conteo_valores.get(val, 0)
                # Suavizado de Laplace: (conteo + alpha) / (n_y + alpha * k)
                tabla.loc[val, y] = (conteo + alpha) / (n_y + alpha * k)
        condicionales[feat] = tabla
    return condicionales

condicionales = calcular_condicionales(train_df, seleccionadas, TARGET, clases, alpha=1)

for feat, tabla in condicionales.items():
    print(f"--- P({feat} | {TARGET}) [Laplace suavizado] ---")
    display(tabla)

--- P(outlook | play) [Laplace suavizado] ---


,no,yes
overcast,0.166667,0.333333
rainy,0.333333,0.333333
sunny,0.500000,0.333333


--- P(humidity | play) [Laplace suavizado] ---


,no,yes
high,0.8,0.375
normal,0.2,0.625


In [53]:
# --- Probabilidad CONJUNTA aproximada: P(Xi, Y) = P(Y) * P(Xi | Y) ---
conjuntas = {}
for feat, tabla_cond in condicionales.items():
    tabla_conjunta = tabla_cond.copy()
    for y in clases:
        tabla_conjunta[y] = tabla_conjunta[y] * prob_marginal[y]
    conjuntas[feat] = tabla_conjunta
    print(f"--- P({feat}, {TARGET}) ---")
    display(tabla_conjunta)

--- P(outlook, play) ---


,no,yes
overcast,0.055556,0.222222
rainy,0.111111,0.222222
sunny,0.166667,0.222222


--- P(humidity, play) ---


,no,yes
high,0.266667,0.250000
normal,0.066667,0.416667


### 2.3 Criterios de relacionamiento entre variables (independencia condicional)

Naive Bayes **no estima directamente la probabilidad conjunta completa** `P(X₁, X₂, …, Xₙ | Y)`, ya que esto requeriría datos suficientes para cada combinación posible de valores (con solo 10 instancias de entrenamiento sería inviable). En su lugar, se explota el supuesto de **independencia condicional entre predictores dada la clase**:

$$P(X_1, X_2, \dots, X_n \mid Y) \approx \prod_{i=1}^{n} P(X_i \mid Y)$$

Esto significa que, **una vez conocida la clase `play`**, el valor de `outlook` no aporta información adicional sobre el valor de `humidity` (u otra variable seleccionada): cada tabla condicional `P(Xᵢ | Y)` estimada en el paso 2.2 se calcula **de forma independiente** por variable, y luego se combinan multiplicativamente durante la inferencia (sección 3). Esta es la simplificación central — y la razón del nombre "naive" (ingenuo) — que hace al modelo computacionalmente tratable incluso con pocos datos.


### Resumen descriptivo de la implementación del modelo probabilístico

El dataset simplificado (`outlook`, `humidity`, `play`) se dividió en **9 instancias de entrenamiento** y **5 de prueba** (70%/30%, semilla fija = 42). A partir del conjunto de entrenamiento se estimó la **probabilidad marginal** de la clase, obteniéndose `P(play = yes) ≈ 0.667` y `P(play = no) ≈ 0.333`, lo que confirma un ligero desbalance hacia la clase "yes" (6 de 9 instancias de entrenamiento).

Sobre esa misma base se calcularon las **probabilidades condicionales** `P(outlook | play)` y `P(humidity | play)` con suavizado de Laplace (α = 1). Estas tablas muestran, por ejemplo, que `P(outlook = overcast | play = yes)` es considerablemente mayor que `P(outlook = overcast | play = no)`, y que `P(humidity = normal | play = yes)` supera claramente a su contraparte para `play = no`, reflejando en las distribuciones estimadas el mismo patrón de asociación detectado en el análisis de correlación (sección 1). A partir de estas condicionales y la marginal se derivaron también las **distribuciones conjuntas aproximadas** `P(Xᵢ, play)` como tablas de referencia.

Todo este cálculo se apoyó explícitamente en el **supuesto de independencia condicional** de Naive Bayes: `outlook` y `humidity` se trataron como variables independientes entre sí una vez conocida la clase, lo que permitió estimar sus distribuciones condicionales por separado (en lugar de una tabla conjunta `P(outlook, humidity | play)`, que con solo 9 instancias de entrenamiento habría sido estadísticamente inviable de estimar de forma directa).


---
## 3. Proceso de inferencia y validación del modelo

### 3.1 Predicción mediante la productoria de probabilidades condicionales

Para cada instancia de prueba, el modelo calcula, para cada clase posible `y`, el **score** (proporcional a la probabilidad posterior):

$$\text{score}(y) = P(Y=y) \prod_{i=1}^{n} P(X_i = x_i \mid Y=y)$$

y predice la clase `ŷ` que **maximiza** dicho score (regla MAP). Por estabilidad numérica, la productoria se calcula en escala logarítmica (suma de log-probabilidades) — matemáticamente equivalente, ya que el logaritmo es una función monótona creciente.


In [54]:
def predecir_instancia(instancia, prob_marginal, condicionales, clases, features):
    # Calcula el score posterior (log) de cada clase y devuelve la clase con mayor score,
    # junto con el detalle del calculo (productoria de condicionales).
    scores_log = {}
    detalle = {}
    for y in clases:
        log_score = np.log(prob_marginal[y])
        pasos = [f"log P({TARGET}={y}) = log({prob_marginal[y]:.4f}) = {np.log(prob_marginal[y]):.4f}"]
        for feat in features:
            valor = instancia[feat]
            tabla_cond = condicionales[feat]
            if valor in tabla_cond.index:
                p_cond = tabla_cond.loc[valor, y]
            else:
                # Valor no visto en entrenamiento -> aplicar suavizado neutro
                k = len(tabla_cond.index)
                p_cond = 1 / (k + 1)
            log_score += np.log(p_cond)
            pasos.append(f"log P({feat}={valor} | {TARGET}={y}) = log({p_cond:.4f}) = {np.log(p_cond):.4f}")
        scores_log[y] = log_score
        detalle[y] = pasos
    y_pred = max(scores_log, key=scores_log.get)
    return y_pred, scores_log, detalle


def predecir_dataset(df_features, prob_marginal, condicionales, clases, features):
    predicciones = []
    for _, fila in df_features.iterrows():
        y_pred, scores_log, _ = predecir_instancia(fila, prob_marginal, condicionales, clases, features)
        predicciones.append(y_pred)
    return predicciones

y_pred_test = predecir_dataset(X_test, prob_marginal, condicionales, clases, seleccionadas)

resultados_test = X_test.copy()
resultados_test['play_real'] = y_test.values
resultados_test['play_predicho'] = y_pred_test
resultados_test


,outlook,humidity,play_real,play_predicho
9,rainy,normal,yes,yes
11,overcast,high,yes,yes
0,sunny,high,no,no
12,overcast,normal,yes,yes
5,rainy,normal,no,yes


### 3.2 Sustento del modelo predictivo mediante la productoria de probabilidades condicionales

A continuación se muestra, para una instancia concreta del conjunto de prueba, el detalle completo del cálculo de la productoria (en escala logarítmica) para cada clase, evidenciando cómo el modelo combina la probabilidad marginal con cada probabilidad condicional para llegar a la decisión final.


In [55]:
idx_ejemplo = X_test.index[0]
instancia_ejemplo = X_test.loc[idx_ejemplo]

y_pred_ej, scores_log_ej, detalle_ej = predecir_instancia(
    instancia_ejemplo, prob_marginal, condicionales, clases, seleccionadas
)

print(f"Instancia de prueba (índice {idx_ejemplo}):")
print(instancia_ejemplo.to_dict())
print(f"Valor real de 'play': {y_test.loc[idx_ejemplo]}")
print()

for y in clases:
    print(f"=== Clase candidata: play = {y} ===")
    for paso in detalle_ej[y]:
        print("  " + paso)
    print(f"  => log-score total = {scores_log_ej[y]:.4f}  (score = {np.exp(scores_log_ej[y]):.6f})")
    print()

print(f"Predicción final (clase con mayor score): play = {y_pred_ej}")

Instancia de prueba (índice 9):
{'outlook': 'rainy', 'humidity': 'normal'}
Valor real de 'play': yes

=== Clase candidata: play = no ===
  log P(play=no) = log(0.3333) = -1.0986
  log P(outlook=rainy | play=no) = log(0.3333) = -1.0986
  log P(humidity=normal | play=no) = log(0.2000) = -1.6094
  => log-score total = -3.8067  (score = 0.022222)

=== Clase candidata: play = yes ===
  log P(play=yes) = log(0.6667) = -0.4055
  log P(outlook=rainy | play=yes) = log(0.3333) = -1.0986
  log P(humidity=normal | play=yes) = log(0.6250) = -0.4700
  => log-score total = -1.9741  (score = 0.138889)

Predicción final (clase con mayor score): play = yes


In [56]:
# --- Evaluación del modelo sobre todo el conjunto de prueba ---
acc = accuracy_score(y_test, y_pred_test)
print(f"Exactitud (accuracy) sobre el conjunto de prueba: {acc:.4f}\n")

print("Matriz de confusión:")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred_test, labels=clases),
    index=[f"real_{c}" for c in clases],
    columns=[f"pred_{c}" for c in clases]
))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_test, labels=clases, zero_division=0))

Exactitud (accuracy) sobre el conjunto de prueba: 0.8000

Matriz de confusión:
          pred_no  pred_yes
real_no         1         1
real_yes        0         3

Reporte de clasificación:
              precision    recall  f1-score   support

          no       1.00      0.50      0.67         2
         yes       0.75      1.00      0.86         3

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5



### 3.3 Resumen descriptivo del proceso de inferencia y validación

El modelo se aplicó a las **5 instancias de prueba** calculando, para cada una, el producto (en escala logarítmica) de la probabilidad marginal y las probabilidades condicionales de `outlook` y `humidity` dado cada valor posible de `play`, y seleccionando la clase con mayor score (regla MAP). En el ejemplo detallado paso a paso (instancia con `outlook = rainy`, `humidity = normal`), el score de `play = yes` (log-score ≈ -1.974, score ≈ 0.139) superó ampliamente al de `play = no` (log-score ≈ -3.807, score ≈ 0.022), llevando al modelo a predecir correctamente `play = yes`, tal como confirma el valor real de esa instancia.

Al validar sobre el conjunto de prueba completo, el modelo alcanzó una **exactitud (accuracy) de 0.80** (4 de 5 predicciones correctas). La matriz de confusión muestra que las 3 instancias reales de `play = yes` fueron clasificadas correctamente en su totalidad (recall = 1.00), mientras que de las 2 instancias reales de `play = no`, solo 1 fue identificada correctamente y la otra se confundió con `play = yes` (recall = 0.50 para la clase "no"). Esto se traduce en una precisión perfecta (1.00) para la clase "no" cuando el modelo la predice, pero una cobertura (recall) limitada para dicha clase, y un desempeño más sólido (precisión 0.75, recall 1.00, F1 ≈ 0.86) para la clase "yes".

En conjunto, estos resultados muestran que el modelo generativo captura razonablemente bien el patrón dominante del dataset (favorecer `play = yes` cuando `outlook` y `humidity` son condiciones favorables), aunque su capacidad para distinguir con precisión los casos de `play = no` es más limitada, en gran medida por el reducido tamaño del conjunto de prueba (solo 5 instancias) y por el desbalance de clases original (9 "yes" vs. 5 "no"). Estas métricas deben interpretarse, por tanto, como una **validación ilustrativa** del funcionamiento del modelo y no como una estimación robusta de desempeño en producción; en un escenario con más datos se recomendaría aplicar **validación cruzada (k-fold)** para obtener una medida más confiable.


---
## Conclusiones de la solución

1. **Selección de variables basada en evidencia estadística:** el uso de la prueba Chi-cuadrado permitió identificar, de manera cuantitativa y no arbitraria, qué variables climáticas están más asociadas con la decisión de jugar, reduciendo la dimensionalidad del problema antes de modelar.

2. **Naturaleza generativa del modelo:** a diferencia de un clasificador discriminativo (que aprendería directamente `P(Y|X)` sin modelar `X`), el enfoque Naive Bayes implementado modela explícitamente cómo se generan las observaciones dentro de cada clase mediante distribuciones marginales y condicionales, y obtiene la predicción aplicando el teorema de Bayes — cumpliendo el rol de un verdadero **agente de aprendizaje generativo**.

3. **El supuesto de independencia condicional es la clave de la eficiencia del modelo:** al asumir que las variables predictoras son independientes entre sí dada la clase, fue posible estimar el modelo de forma confiable incluso con un dataset de apenas 14 instancias, algo que sería imposible si se intentara estimar la distribución conjunta completa sin factorizar.

4. **El suavizado de Laplace es indispensable en datasets pequeños:** sin él, cualquier combinación de valor-clase no observada en entrenamiento asignaría probabilidad cero a toda la productoria, invalidando la predicción. Su inclusión hace al modelo robusto frente a la escasez de datos.

5. **Limitaciones:** el tamaño reducido del dataset limita la robustez estadística tanto del análisis de correlación (Chi-cuadrado con frecuencias esperadas bajas) como de la validación (pocas instancias de prueba). Para un despliegue real, se recomendaría recolectar más datos y/o emplear validación cruzada k-fold para obtener estimaciones de desempeño más confiables.

6. **Aplicabilidad general:** el flujo de trabajo desarrollado (correlación → selección de variables → estimación de distribuciones → inferencia por productoria) es **generalizable** a cualquier problema de clasificación con variables categóricas, constituyendo una plantilla reutilizable para agentes de aprendizaje generativo basados en Naive Bayes.
